## PaperUno_Figure_MS

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib import gridspec, ticker
from matplotlib.colorbar import Colorbar
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, to_rgba
from scipy import ndimage

In [ ]:
# Ensure custom module Path is set before import
sys.path.append(r"D:\CHENG\OneDrive - UAB\ICMAB-Python\Figure")
from colors import tol_cmap, tol_cset  # type: ignore

# 画图的初始设置
plt.style.use(r"D:\CHENG\OneDrive - UAB\ICMAB-Python\Figure\liuchzzyy.mplstyle")
# print(plt.style.available)  # noqa: ERA001

# xarray setting
xr.set_options(
    cmap_sequential="viridis",
    cmap_divergent="viridis",
    display_width=150,
)  # viridis, gray

# 颜色设定
colors = tol_cset("vibrant")
if colors is not None:
    colors = list(colors)
else:
    # Fallback colors in case tol_cset returns None
    colors = ["#0077BB", "#33BBEE", "#009988", "#EE7733", "#CC3311", "#EE3377", "#BBBBBB"]
if r"sunset" not in plt.colormaps():
    cmap = tol_cmap("sunset")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)
if r"rainbow_PuRd" not in plt.colormaps():
    cmap = tol_cmap("rainbow_PuRd")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)  # 备用 plasma

# 输出的文件夹
path_out = Path(r"C:\Users\chengliu\Desktop\Figure")

# Set math font
mpl.rcParams["mathtext.fontset"] = "custom"
mpl.rcParams["mathtext.rm"] = "Arial"
mpl.rcParams["mathtext.it"] = "Arial:italic"
mpl.rcParams["mathtext.bf"] = "Arial:bold"
mpl.rcParams["mathtext.sf"] = "Arial"
mpl.rcParams["mathtext.tt"] = "Arial"
mpl.rcParams["mathtext.cal"] = "Arial"
mpl.rcParams["mathtext.default"] = "regular"

### Figure TOC

In [ ]:
# opEXAFS 的数据
path_exafs = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\Overview"
)
exafs_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_exafs, r"cell3_opXAS_P2a_Mn_Oct2022_Rkbg_1.3_Powder.chir2_mag"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

exafs_Mn = exafs_Mn.iloc[:, 0:14]  # FD 数据  # noqa: N816

In [ ]:
# 画图
fig = plt.figure(figsize=(4.67, 5))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.05, hspace=0.0, figure=fig)

xas_colors = [[colors[0], colors[4]], [colors[2], colors[1]]] # type: ignore
labels = [[r"0.2M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
xas_cmap = [
    ListedColormap(
        mpl.colormaps["coolwarm"](np.linspace(0.5, 1.0, exafs_Mn.shape[1]-3)),
        name="xas_cmap_Mn",
    ),
    ListedColormap(
        mpl.colormaps["coolwarm"](np.linspace(0.5, 0.0, exafs_Mn.shape[1]-3)),
        name="xas_cmap_Zn",
    ),
]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.6)

for i in range(exafs_Mn.shape[1] - 3):
    ax.plot(exafs_Mn.iloc[:, 0], exafs_Mn.iloc[:, i + 3], c=xas_cmap[0].colors[i], ls="-", lw=1.0) # type: ignore

ax.set_xlim(0, 6)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.set_ylim(0, 1.6)
ax.set_ylabel(r"FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axvline(x=2.40, ymin=0.0, ymax=0.8, ls="--", lw=1.0, color="k", alpha=0.5)
ax.axvline(x=3.01, ymin=0.0, ymax=0.3, ls="--", lw=1.0, color="k", alpha=0.5)


ax.text(1.3, 1.1, r"Mn-O", transform=ax.transData, fontsize=8, va="center", ha="right", fontfamily="Arial")
ax.text(
    2.0,
    1.4,
    r"$\mathrm{Mn-Mn_{edge}}$",
    transform=ax.transData,
    fontsize=8,
    va="center",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    2.8,
    0.6,
    r"$\mathrm{Mn-Mn_{corner}}$",
    transform=ax.transData,
    fontsize=8,
    va="center",
    ha="left",
    fontfamily="Arial",
)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.04, 0.0, 0.05, 0.3)),
    location="right",
    orientation="vertical",
    cmap=xas_cmap[0].reversed(),
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False) # type: ignore
ax.text(
    1.10, 0.30, r"Discharge", transform=ax.transAxes, fontsize=9, va="top", ha="left", fontfamily="Arial", rotation=270
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_TOC_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_TOC_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_TOC_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_TOC_900.svg"),
    transparent=False,
    pad_inches=0.01,
    bbox_inches="tight",
    dpi=900,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure 5

In [ ]:
# edge-corner model, # DFT
edge_corner_model = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\DFT\αMnO2\Results\Edge_Corner.tif")  # noqa: RUF001
# Saivash Data
path_saivash = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\ExSitu\αMnO2\XAS\EMD-SK")  # noqa: RUF001
# EXAFS 数据
emd_exafs = pd.read_csv(
    Path.joinpath(path_saivash, r'Results', r"EMD_Electrode.chir2_mag"), sep=r"\s+", comment="#", index_col=None,
    header=None,
).dropna(axis=1, how="all")
emd_exafs.columns = ["r", r"alpha_MnO2_eletrode", r"HAC_fc_MnK"]

# 读取电化学数据
path_filelist = list(Path.joinpath(path_saivash, r"Echem", r"HaC FC").glob(r"*.txt"))
echem = []
for path_file in path_filelist[1:2]:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    df = df.iloc[:, [0, 1, 2, 4, 5]]
    echem.append(df)

# 读取一个常规的 αMnO2 的数据
path_filelist = list(
    Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx")
    )

# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
        axis=1, how="all"
    )
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i].iloc[:, 0].isin([0, 1])]
    echem[i] = echem[i][echem[i].iloc[:, 2] >= 0]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(3.3, 4))
gs = gridspec.GridSpec(2, 2, width_ratios=[1.5, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2])
ax = subfig.add_axes((0, 0, 1.0, 1.0))

im = plt.imread(edge_corner_model)
ax.imshow(im, aspect="equal")
ax.set_axis_off()

# ax.text(
#     0.0, 1.0, r"a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
# )  # noqa: ERA001

# 图 B
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{EMD \ MnO_2}$", r"$\mathrm{\alpha-MnO_2}$"]]
for i, data in enumerate(echem[0:1]):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        temp = temp[temp["Ewe/V"] > 0.9]
        # 找到容量的最大值的索引
        idx_max = temp["Capacity/mA.h"].idxmax()
        # 断开最大值前后的曲线
        ax.plot(
            temp.loc[idx_max + 1 : 442, "Capacity/mA.h"] / temp.loc[idx_max, "Capacity/mA.h"] * 100,
            temp.loc[idx_max + 1 : 442, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[1], # type: ignore
            label=labels[i][j],
            zorder=0,
        )
        CE_EMD = (temp.loc[idx_max + 1 : 442, "Capacity/mA.h"] / temp.loc[idx_max, "Capacity/mA.h"]).max() * 100

for i, data in enumerate(echem[1:2]):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"] / 5.57,
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0], # type: ignore
            label=labels[i][j + 1],
            zorder=0,
        )
        CE_αMnO2 = (temp.loc[: idx_min - 1, "SpeCap/mAh/g"] / 557).max() * 100  # noqa: PLC2401

# 设置刻度线等格式
ax.set_xlabel(r"Percentage of Full Charge", fontsize=9, labelpad=2.0)
ax.set_xlim(0, 100.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"$\mathrm{Voltage \ (V \ vs. \ Zn/Zn^{2+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.7, 2.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=-0.1))

ax.tick_params(
    axis="both", which="both", direction="out", labelsize=9, top=False, right=False, labeltop=False, labelright=False
)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    0.9,
    0.1,
    f"CE:{CE_EMD:.1f}%",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    color=colors[1], # type: ignore
)
ax.text(
    0.55,
    0.1,
    f"CE:{CE_αMnO2:.1f}%",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    color=colors[0], # type: ignore
)

# ax.text(-0.23, 1.0, r'b', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')  # noqa: E501, ERA001

# 图 C
subfig = fig.add_subfigure(gs[1, 1])
ax = subfig.add_axes((0.6, 0, 1.0, 1.0))
ax.set_box_aspect(1.2)

for i in range(emd_exafs.shape[1] - 1):
    ax.plot(emd_exafs.iloc[:, 0], emd_exafs.iloc[:, i + 1], c=colors[i], label=labels[0][i], ls="-", lw=1.0) # type: ignore
ax.set_xlim(0, 4)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9, labelpad=2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))

ax.set_ylim(0, 2.4)
ax.set_ylabel(r"FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9, labelpad=2.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
# ax.legend(loc='upper left', bbox_to_anchor=(0.5, 1), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)  # noqa: E501, ERA001
ax.text(
    0.95,
    0.9,
    r"edge-sharing intensity (%)",
    transform=ax.transAxes,
    fontsize=7,
    va="center",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.95, 0.8, r"85.0", transform=ax.transAxes, fontsize=7, va="center", ha="right", fontfamily="Arial", color=colors[1] # type: ignore
)
ax.text(
    0.95, 0.7, r"74.4", transform=ax.transAxes, fontsize=7, va="center", ha="right", fontfamily="Arial", color=colors[0] # type: ignore
)

# ax.text(-0.4, 1.0, r'c', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')  # noqa: E501, ERA001

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_05_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_05_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_05_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_05_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure 4. sTXM Identifications of Mn(III) species at the α-MnO2 nanowire surface

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, scale, unit, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / scale,
        "{} {}".format(size, unit),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=1,
        frameon=False,
        color=color,
        label_top=True,
    )
    scalebar = ax.add_artist(asb)
    return scalebar


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


def index(energy, value):
    return np.argmin(abs(energy - value))

In [ ]:
# sTXM 的数据
path_sTXM_Mn = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\STXM\ExSitu\Andrea")  # noqa: N816
sTXM_Mn_raw = pd.read_csv(  # noqa: N816
    Path.joinpath(path_sTXM_Mn, r"SpectrumAll", r"sTXM_PristVSB6_Larua.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
arctan_mapping_pristine = pd.read_csv(
    Path.joinpath(path_sTXM_Mn, r'αMnO2', r"Pristine", r"E1 Pristine", r"Pristine_Im_ratio2.txt"),  # noqa: RUF001
    sep=r",",
    index_col=None,
    header=None,
    comment="#",
)
arctan_mapping_1stDis = pd.read_csv(  # noqa: N816
    Path.joinpath(path_sTXM_Mn, r'αMnO2', r"Charge", r"B6 1st0.9V", r"B6_Imratio2.txt"),  # noqa: RUF001
    sep=r",",
    index_col=None,
    header=None,
    comment="#",
)
arctan_mapping_MnOOH = pd.read_csv(  # noqa: N816
    Path.joinpath(path_sTXM_Mn, r'MnOOH', r"Pristine", r"F8 MnOOH", r"MnOOH_Imratio2.txt"),
    sep=r",",
    index_col=None,
    header=None,
    comment="#",
)
mcr = pd.read_excel(
    Path.joinpath(path_sTXM_Mn, r"SpectrumAll", r"sTXM_MCR-ALS-3comp.xlsx"),
    sheet_name="Concentrations",
    index_col=0,
    header=0,
    comment="#",
    engine="openpyxl",
)

In [ ]:
energy = sTXM_Mn_raw.iloc[:, 0].values
arctan_mapping_pristine.fillna(0, inplace=True)
arctan_mapping_1stDis.fillna(0, inplace=True)

arctan_mapping_pristine1 = arctan_mapping_pristine.copy(deep=True)
arctan_mapping_1stDis1 = arctan_mapping_1stDis.copy(deep=True)  # noqa: N816
arctan_mapping_MnOOH1 = arctan_mapping_MnOOH.copy(deep=True)  # noqa: N816

arctan_mapping_pristine.where(arctan_mapping_pristine <= 0.551, 0, inplace=True)  # noqa: PLR2004
arctan_mapping_pristine.where(arctan_mapping_pristine >= 0.20, 0, inplace=True)  # noqa: PLR2004

arctan_mapping_1stDis.where(arctan_mapping_1stDis <= 0.551, 0, inplace=True)  # noqa: PLR2004
arctan_mapping_1stDis.where(arctan_mapping_1stDis >= 0.20, 0, inplace=True)  # noqa: PLR2004

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(7, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)
vmin = arctan_mapping_pristine.min().min()  # arctan_mapping_pristine.min().min(), 0.1
vmax = arctan_mapping_pristine.max().max()  # arctan_mapping_pristine.max().max(), 0.7

im = ax.imshow(arctan_mapping_pristine, vmin=vmin, vmax=vmax, cmap="hot")
scalebar = add_sizebar(ax=ax, size=2, scale=0.013000000268220901, unit="um", color="w")
scalebar.set_bbox_to_anchor((0, 0.03), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)

cax = subfig.add_subplot()
cax.set_position((1.03, 0.06, 0.06, 0.6))
cbar = subfig.colorbar(
    mappable=im, cax=cax, ticks=np.linspace(0, 1, 21), location="right", orientation="vertical", format="%.2f"
)
cbar.ax.set_ylim(0.20, vmax)
cax.tick_params(axis="x", direction="out")

ax.text(
    -0.12,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0, 1.0, 1.0))
ax.set_box_aspect(1.0)

vmin = arctan_mapping_1stDis.min().min()  # arctan_mapping_1stDis.min().min(), 0.1
vmax = arctan_mapping_1stDis.max().max()  # arctan_mapping_1stDis.max().max(), 0.7

im = ax.imshow(arctan_mapping_1stDis, vmin=vmin, vmax=vmax, cmap="hot")
scalebar = add_sizebar(ax=ax, size=2, scale=0.013000000268220901, unit="um", color="w")
scalebar.set_bbox_to_anchor((0, 0.03), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)

cax = subfig.add_subplot()
cax.set_position((1.33, 0.06, 0.06, 0.6))
cbar = subfig.colorbar(
    mappable=im, cax=cax, ticks=np.linspace(0, 1, 21), location="right", orientation="vertical", format="%.2f"
)
cax.tick_params(axis="x", direction="out")
cbar.ax.set_ylim(0.20, vmax)
ax.text(
    -0.12,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.025, 1.0, 1.0))
ax.set_box_aspect(0.85)

arctan_mapping_pristine_b = arctan_mapping_pristine1.where(arctan_mapping_pristine1 > 0.05, np.nan, inplace=False)  # noqa: PLR2004
arctan_mapping_1stDis_b = arctan_mapping_1stDis1.where(arctan_mapping_1stDis1 > 0.05, np.nan, inplace=False)  # noqa: N816, PLR2004
arctan_mapping_MnOOH_b = arctan_mapping_MnOOH1.where(arctan_mapping_MnOOH1 > 0.05, np.nan, inplace=False)  # noqa: N816, PLR2004


ax.hist(
    arctan_mapping_pristine_b.values.flatten(),
    bins=100,
    density=True,
    color=colors[0], # type: ignore
    align="mid",
    range=(vmin, vmax),
    label=r"Pristine",
    stacked=True,
)
ax.hist(
    arctan_mapping_1stDis_b.values.flatten(),
    bins=100,
    density=True,
    color=colors[1], # type: ignore
    align="mid",
    range=(vmin, vmax),
    label=r"Discharged",
    alpha=0.7,
    stacked=True,
)
ax.hist(
    arctan_mapping_MnOOH_b.values.flatten(),
    bins=100,
    density=True,
    color=colors[3], # type: ignore
    align="mid",
    range=(0.1, 1.1),
    label=r"Ref.MnOOH",
    alpha=0.7,
    stacked=True,
)

ax.set_ylabel(r"Normalized Frequency", fontsize=11)
ax.set_xlabel(
    r"Shape Index",
    fontsize=11,
)
ax.set_xlim(0.2, 1.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_ylim(0, 8.5)
# ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))  # noqa: ERA001
# ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))  # noqa: ERA001

ax.tick_params(
    axis="y",
    which="both",
    left=False,
    labelbottom=False,
    labelleft=False,
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.5, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handletextpad=0.3,
    labelspacing=0.3,
)
# ax.annotate(" ", xy=(0.43, 0.64), xycoords='axes fraction',xytext=(0.36, 0.97), textcoords='axes fraction', size=8, va="center", ha="center",  arrowprops=dict(arrowstyle="->", ls='--', color='k',connectionstyle="arc3,rad=0.1"))  # noqa: E501, ERA001
ax.text(
    -0.12,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0.025, 1.0, 1.0))
ax.set_box_aspect(0.85)
ax.set_facecolor("k")
cmap_sTXM_Mn = ListedColormap(  # noqa: N816
    mpl.colormaps["hot"](np.linspace(0.15 / (vmax - vmin), 0.55 / (vmax - vmin), sTXM_Mn_raw.values.shape[1] // 2 - 6)),
    name="cmap_sTXM_Mn",
)

for i in range(sTXM_Mn_raw.values.shape[1] // 2 - 6):
    ax.plot(energy, sTXM_Mn_raw.iloc[:, i + 3].values + 1.5, c=cmap_sTXM_Mn.colors[i], ls="-", lw=0.8) # type: ignore
    # print(i+1)  # noqa: ERA001
for i in range(sTXM_Mn_raw.values.shape[1] // 2 - 6):
    ax.plot(energy, sTXM_Mn_raw.iloc[:, i + 17].values, c=cmap_sTXM_Mn.colors[i], ls="-", lw=0.8) # type: ignore
    # print(i+16)  # noqa: ERA001

# ax.plot(energy, sTXM_Mn_raw.iloc[:,1].values+2, c='w', ls='--', lw=0.8, alpha=0.3)  # noqa: ERA001
# ax.plot(energy, sTXM_Mn_raw.iloc[:,2].values+2, c='w', ls='--', lw=0.8, alpha=0.3)  # noqa: ERA001
# for i in range(sTXM_Mn_raw.values.shape[1]//2-8):
#     ax.plot(energy, sTXM_Mn_raw.iloc[:,i+10].values+2, c='w', ls='--', lw=0.8, alpha=0.3)  # noqa: ERA001

# ax.plot(energy, sTXM_Mn_raw.iloc[:,15].values, c='w', ls='--', lw=0.8, alpha=0.3)  # noqa: ERA001
# ax.plot(energy, sTXM_Mn_raw.iloc[:,16].values, c='w', ls='--', lw=0.8, alpha=0.3)  # noqa: ERA001
# for i in range(sTXM_Mn_raw.values.shape[1]//2-9):
#     ax.plot(energy, sTXM_Mn_raw.iloc[:,i+23].values, c='w', ls='--', lw=0.8, alpha=0.3)  # noqa: ERA001

ax.set_xlim(639.0, 649.0)
ax.set_xlabel(r"Energy (eV)", fontsize=11, labelpad=3)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=-1.0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=-1.0))
ax.set_ylim(0, 5)
ax.set_ylabel(r"Normalized absorption (arb.u.)", fontsize=11)
# ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1))  # noqa: ERA001
# ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5)) # noqa: ERA001
ax.tick_params(axis="x", labelsize=9)
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(
    axis="y",
    which="both",
    left=False,
    labelbottom=False,
    labelleft=False,
)
ax.text(0.95, 0.9, r"Pristine", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="w")
ax.text(
    0.95, 0.07, r"Discharged", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="w"
)
# # Add colorbar and adjust ticks afterwards
# colorbar = Colorbar(ax=ax.inset_axes([0.03, 0.8, 0.4, 0.06]), location='bottom', orientation='horizontal',
#                       cmap=cmap_sTXM_Mn, ticklocation='top', spacing = 'proportional', drawedges=False)
# colorbar.set_ticks([]) # noqa: ERA001

# ax.text(0.03, 0.95, f'{vmin:.2f}', transform=ax.transAxes, fontsize=8,  va='top', ha='left', fontfamily='Arial', )  # noqa: E501, ERA001
# ax.text(0.31, 0.95, f'{vmax:.2f}', transform=ax.transAxes, fontsize=8, va='top', ha='left', fontfamily='Arial', )  # noqa: E501, ERA001

ax.text(
    -0.12,
    1.0,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[0:2, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.85, 0.065, 0.6, 0.9))
# ax.set_box_aspect(2.05)  # noqa: ERA001
ax.plot(
    np.linspace(0, 1, num=mcr.iloc[0:5, 0].shape[0], endpoint=True), mcr.iloc[0:5, 0], label=r"Pristine", c=colors[0] # type: ignore
)
ax.plot(np.linspace(0, 1, num=mcr.iloc[0:5, 0].shape[0], endpoint=True), mcr.iloc[0:5, 1], label=None, c=colors[1]) # type: ignore
ax.plot(np.linspace(0, 1, num=mcr.iloc[0:5, 0].shape[0], endpoint=True), mcr.iloc[0:5, 2], label=None, c=colors[5]) # type: ignore

ax.plot(
    np.linspace(0, 1, num=mcr.iloc[10:15, 0].shape[0], endpoint=True),
    mcr.iloc[10:15, 0],
    label=r"Discharged",
    ls="--",
    c=colors[0], # type: ignore
)
ax.plot(
    np.linspace(0, 1, num=mcr.iloc[10:15, 0].shape[0], endpoint=True),
    mcr.iloc[10:15, 1],
    label=None,
    ls="--",
    c=colors[1], # type: ignore
)
ax.plot(
    np.linspace(0, 1, num=mcr.iloc[10:15, 0].shape[0], endpoint=True),
    mcr.iloc[10:15, 2],
    label=None,
    ls="--",
    c=colors[5], # type: ignore
)
ax.set_ylabel(r"Relative Concentrations (arb.u.)", fontsize=11)
ax.set_xlabel("", fontsize=11)
# ax.set_xlim(0, 0.8)  # noqa: ERA001
ax.set_ylim(-0.02, 1.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.tick_params(
    axis="x",
    which="both",
    bottom=False,
    top=False,
    labelbottom=False,
    labeltop=False,
)
ax.text(
    0,
    -0.05,
    r"Bulk",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.7,
    -0.05,
    r"Border",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.4,
    0.7,
    r"Mn(IV)",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    c=colors[0], # type: ignore
)
ax.text(
    0.4,
    0.3,
    r"Mn(III)",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    c=colors[1], # type: ignore
)
ax.text(
    0.9,
    0.12,
    r"Mn (II/III)",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    c=colors[5], # type: ignore
)

ax.text(
    -0.35,
    1.0,
    r"e",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

legend = ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.2, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="k",
    fontsize=9,
    columnspacing=0.5,
    handletextpad=0.5,
    labelspacing=0.3,
)
legend.legend_handles[0].set_color("k") # type: ignore
legend.legend_handles[0].set_markerfacecolor("w") # type: ignore
legend.legend_handles[0].set_markeredgecolor("k") # type: ignore
legend.legend_handles[1].set_color("k") # type: ignore
legend.legend_handles[1].set_markerfacecolor("w") # type: ignore
legend.legend_handles[1].set_markeredgecolor("k") # type: ignore

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_04_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_04_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_04_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_04_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure 3: α-MnO2 dissolution in depth

In [ ]:
# opEXAFS 的数据
path_exafs = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\Overview"
)
exafs_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_exafs, r"cell3_opXAS_P2a_Mn_Oct2022_Rkbg_1.3_Powder.chir2_mag"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
exafs_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_exafs, r"cell3_opXAS_P2a_Zn_Oct2022_Rkbg_1.1.chir2_mag"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

# edge - corner 之间的面积比值
path_file = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\FD1st\Oct2022_cell3_P2a\EXAFS\Mn\EXAFS_intensity.csv"  # noqa: E501
)
intensity_ratio = pd.read_csv(path_file, sep=",", index_col=None, header=0, comment="#")

# Pre Edge 的数据
path_edge = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\FD1st\Oct2022_cell3_P2a\PreEdge\Version_4"  # noqa: E501
)
opPreEdge = pd.read_csv(Path.joinpath(path_edge, r"PreEdge.csv"), sep=",", comment="#", header=0, index_col=None)  # noqa: N816
Mn_2 = xr.open_dataset(Path.joinpath(path_edge, r"3-Fittings", r"Mn_2_result_voigt_2.NETCDF4"), engine="h5netcdf")
Mn_pristine = xr.open_dataset(
    Path.joinpath(path_edge, r"3-Fittings", r"Prisitne_Mn_4_result_ref_2.NETCDF4"), engine="h5netcdf"
)
Mn_FD1st_ex = xr.open_dataset(
    Path.joinpath(path_edge, r"3-Fittings", r"FD1st_ex_result_ref_2.NETCDF4"), engine="h5netcdf"
)

In [ ]:
intensity_ratio = intensity_ratio.iloc[:11, :]
exafs_Mn = exafs_Mn.iloc[:, 0:14]  # FD 数据  # noqa: N816
exafs_Zn = exafs_Zn.iloc[:, 0:14]  # FD 数据  # noqa: N816

In [ ]:
# 画图
fig = plt.figure(figsize=(4.67, 5))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.05, hspace=0.0, figure=fig)

xas_colors = [[colors[0], colors[4]], [colors[2], colors[1]]] # type: ignore
labels = [[r"0.2M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
xas_cmap = [
    ListedColormap(
        mpl.colormaps["coolwarm"](np.linspace(0.5, 1.0, opPreEdge.shape[1] // 4 - 6)),
        name="xas_cmap_Mn",
    ),
    ListedColormap(
        mpl.colormaps["coolwarm"](np.linspace(0.5, 0.0, opPreEdge.shape[1] // 4 - 6)),
        name="xas_cmap_Zn",
    ),
]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(2):
    ax.plot(
        exafs_Mn.iloc[:, 0],
        exafs_Mn.iloc[:, i + 1] + 1.6,
        c=xas_colors[0][i],
        ls="--",
        lw=1.0,
        label=labels[0][i],
        zorder=5,
    )
for i in range(exafs_Mn.shape[1] - 3):
    ax.plot(exafs_Mn.iloc[:, 0], exafs_Mn.iloc[:, i + 3], c=xas_cmap[0].colors[i], ls="-", lw=1.0) # type: ignore

ax.set_xlim(0, 6)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.set_ylim(0, 4.2)
ax.set_ylabel(r"FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))

ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axvline(x=2.40, ymin=0.0, ymax=0.7, ls="--", lw=1.0, color="k", alpha=0.5)
ax.axvline(x=3.01, ymin=0.0, ymax=0.5, ls="--", lw=1.0, color="k", alpha=0.5)

# arrowprops = dict(arrowstyle="<-", color='k',
#     connectionstyle="angle,angleA=90,angleB=0,rad=20")
# ax.annotate(r'Mn-O', fontsize=8, xy=(1.4, 1.15), xycoords='data', xytext=(0.02, 0.1), textcoords='axes fraction', arrowprops=arrowprops) # noqa: E501, ERA001
# arrowprops = dict(arrowstyle="<-", color='k',
#     connectionstyle="angle,angleA=90,angleB=0,rad=20")
# ax.annotate(r'$\mathrm{Mn-Mn_{edge}}$', fontsize=8, xy=(2.42, 1.3), xycoords='data', xytext=(0.75, 0.15), textcoords='axes fraction', arrowprops=arrowprops) # noqa: E501, ERA001
# arrowprops = dict(arrowstyle="<-", color='k',)  # noqa: ERA001
# ax.annotate(r'$\mathrm{Mn-Mn_{corner}}$', fontsize=8, xy=(3.0, 0.45), xycoords='data', xytext=(0.45, 0.22), textcoords='axes fraction', arrowprops=arrowprops)  # noqa: E501, ERA001

ax.text(1.4, 1.3, r"Mn-O", transform=ax.transData, fontsize=8, va="center", ha="right", fontfamily="Arial")
ax.text(
    2.6,
    1.4,
    r"$\mathrm{Mn-Mn_{edge}}$",
    transform=ax.transData,
    fontsize=8,
    va="center",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    3.0,
    0.7,
    r"$\mathrm{Mn-Mn_{corner}}$",
    transform=ax.transData,
    fontsize=8,
    va="center",
    ha="left",
    fontfamily="Arial",
)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.04, 0.0, 0.05, 0.3)),
    location="right",
    orientation="vertical",
    cmap=xas_cmap[0].reversed(),
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False) # type: ignore
ax.text(
    1.10, 0.30, r"Discharge", transform=ax.transAxes, fontsize=9, va="top", ha="left", fontfamily="Arial", rotation=270
)

ax.text(
    -0.25,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 插入图片
ax2 = ax.inset_axes((0.65, 0.68, 0.6, 0.3))
# ax2.set_box_aspect(0.6)  # noqa: ERA001

capacity = np.linspace(0, 270, 11)
ax2.plot(capacity, intensity_ratio.iloc[:, 1], ls="-", marker="o", ms=4, lw=1.0, c=colors[0], label=r"$\mathrm{Edge}$") # type: ignore
ax2.plot(
    capacity, intensity_ratio.iloc[:, 2], ls="-", marker="o", ms=4, lw=1.0, c=colors[1], label=r"$\mathrm{Corner}$" # type: ignore
)

ax2.set_xlim(-10, 300)
ax2.set_xlabel(r"Capacity (mAh g$\mathrm{^{-1}_{MnO_2}}$)", fontsize=7, labelpad=0)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30))
ax2.set_ylim(0.0, 1.5)
ax2.set_ylabel(r"FTs Intensity", fontsize=7)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))
ax2.legend(
    handlelength=1.5,
    loc="upper left",
    bbox_to_anchor=(0.4, 1.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=7,
    labelspacing=0.1,
)

ax2.tick_params(axis="both", which="both", labelsize=7, bottom=True, left=True, labelbottom=True, labelleft=True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.1, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(2):
    ax.plot(
        exafs_Zn.iloc[:, 0],
        exafs_Zn.iloc[:, i + 1] + 0.7,
        c=xas_colors[1][i],
        ls="--",
        lw=1.0,
        label=labels[1][i],
        zorder=5,
    )
for i in range(exafs_Zn.shape[1] - 3):
    ax.plot(exafs_Zn.iloc[:, 0], exafs_Zn.iloc[:, i + 3], c=xas_cmap[1].colors[i], ls="-", lw=1.0) # type: ignore

ax.set_xlim(0, 6)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.set_ylim(0, 2.0)
ax.set_ylabel(r"FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))
ax.legend(loc="upper left", bbox_to_anchor=(0.5, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.04, 0.0, 0.05, 0.3)),
    location="right",
    orientation="vertical",
    cmap=xas_cmap[1].reversed(),
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False) # type: ignore
ax.text(
    1.10, 0.30, r"Discharge", transform=ax.transAxes, fontsize=9, va="top", ha="left", fontfamily="Arial", rotation=270
)

ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axvline(x=2.40, ymin=0.0, ymax=0.5, lw=1.0, ls="--", color="k", alpha=0.5)
ax.axvline(x=3.01, ymin=0.0, ymax=0.5, lw=1.0, ls="--", color="k", alpha=0.5)

# arrowprops = dict(arrowstyle="<-", color='k',
#     connectionstyle="angle,angleA=90,angleB=0,rad=20")
# ax.annotate(r'Zn-O', fontsize=8, xy=(1.6, 1.9), xycoords='data', xytext=(0.02, 0.8), textcoords='axes fraction', arrowprops=arrowprops)  # noqa: E501, ERA001
# arrowprops = dict(arrowstyle="<-", color='k',
#     connectionstyle="angle,angleA=0,angleB=90,rad=20")
# ax.annotate(r'Zn-Zn', fontsize=8, xy=(2.65, 1.1), xycoords='data', xytext=(0.6, 0.6), textcoords='axes fraction', arrowprops=arrowprops)  # noqa: E501, ERA001
ax.text(1.8, 1.95, r"Zn-O", transform=ax.transData, fontsize=8, va="center", ha="right", fontfamily="Arial")
ax.text(2.65, 1.15, r"Zn-Zn", transform=ax.transData, fontsize=8, va="center", ha="center", fontfamily="Arial")

ax.text(
    -0.25,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
left = 0.65
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((left, -0.34, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.plot(opPreEdge.iloc[:, 4] + 0.35, opPreEdge.iloc[:, 7], ls="-", color=colors[4], label=r"Ref.MnO") # type: ignore
ax.plot(opPreEdge.iloc[:, 0], opPreEdge.iloc[:, 3], ls="-", color=colors[5], label=r"Ref.Mn$\mathrm{_2O_3}$") # type: ignore
ax.plot(opPreEdge.iloc[:, 8], opPreEdge.iloc[:, 11], ls="-", color=colors[3], label=r"Ref.Mn$\mathrm{O_2}$") # type: ignore

ax.set_xlim(6535.0, 6550.0)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=3, offset=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=1))
ax.set_ylim(-0.005, 0.10)
ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=9, labelpad=8)
ax.yaxis.set_label_coords(-0.18, 3.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.05))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.025))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.7, 1.15),
    ncols=1,
    frameon=False,
    fontsize=9,
    labelcolor="linecolor",
    columnspacing=0.0,
)
ax.tick_params(
    axis="both", which="both", direction="out", labelsize=9, left=True, labelleft=True, bottom=True, labelbottom=True
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.axvline(x=6540.3, ymax=0.9, ls='--', color='k', alpha=0.5)  # noqa: ERA001
ax.axvline(x=6540.8, ymax=0.9, ls="--", color="k", alpha=0.5)
ax.axvline(x=6543.0, ymax=0.9, ls="--", color="k", alpha=0.5)
ax.axvline(x=6541.3, ymax=0.9, ls="--", color="k", alpha=0.5)

# 图 C1
ax = subfig.add_subplot()
ax.set_position((left, 0.03, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.plot(Mn_pristine["aMnO2"]["Energy_aMnO2"], Mn_pristine["aMnO2"], ls="-", color=colors[3], label="Prisitne") # type: ignore
ax.fill_between(
    Mn_pristine["fit_Mn_III"]["Energy_fit"],
    y1=Mn_pristine["fit_Mn_III"],
    y2=0,
    ls="-",
    color=colors[5], # type: ignore
    label=None,
    alpha=0.5,
)
ax.fill_between(
    Mn_pristine["fit_Mn_IV"]["Energy_fit"],
    y1=Mn_pristine["fit_Mn_IV"],
    y2=0,
    ls="-",
    color=colors[3], # type: ignore
    label=None,
    alpha=0.5,
)
ax.plot(Mn_pristine["fit"]["Energy_fit"], Mn_pristine["fit"], ls="--", color=colors[1], label=None) # type: ignore

ax.set_xlim(6535.0, 6550.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylim(0.0, 0.08)
# ax.set_ylabel(r'Relative Intensity (arb.u.)', fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.03, offset=-0.01))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.015, offset=-0.01))

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.7, 1.0),
    ncols=1,
    frameon=False,
    fontsize=9,
    labelcolor="linecolor",
    columnspacing=0.0,
)
ax.tick_params(
    axis="both", which="both", direction="out", labelsize=9, left=True, labelleft=True, bottom=False, labelbottom=False
)
ax.text(
    0.05,
    1.0,
    r"Mn(III) = 27.0 at.%",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    alpha=0.5,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
# ax.axvline(x=6540.3, ymax=0.8, ls='--', color='k', alpha=0.5)  # noqa: ERA001
ax.axvline(x=6540.8, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.axvline(x=6543.0, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.axvline(x=6541.3, ymax=0.8, ls="--", color="k", alpha=0.5)

# 图 C2
ax = subfig.add_subplot()
ax.set_position((left, 0.45, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.plot(Mn_FD1st_ex["aMnO2"]["Energy_aMnO2"], Mn_FD1st_ex["aMnO2"], ls="-", color=colors[0], label="Discharge") # type: ignore
ax.fill_between(
    Mn_FD1st_ex["fit_Mn_III"]["Energy_fit"],
    y1=Mn_FD1st_ex["fit_Mn_III"],
    y2=0,
    ls="-",
    color=colors[5], # type: ignore
    label=None,
    alpha=0.5,
)
ax.fill_between(
    Mn_FD1st_ex["fit_Mn_IV"]["Energy_fit"],
    y1=Mn_FD1st_ex["fit_Mn_IV"],
    y2=0,
    ls="-",
    color=colors[3], # type: ignore
    label=None,
    alpha=0.5,
)
ax.plot(Mn_FD1st_ex["fit"]["Energy_fit"], Mn_FD1st_ex["fit"], ls="--", color=colors[1], label=None) # type: ignore

ax.set_xlim(6535.0, 6550.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylim(0.0, 0.08)
# ax.set_ylabel(r'Relative Intensity (arb.u.)', fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.03, offset=-0.01))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.015, offset=-0.01))

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.7, 1.0),
    ncols=1,
    frameon=False,
    fontsize=9,
    labelcolor="linecolor",
    columnspacing=0.0,
)
ax.tick_params(
    axis="both", which="both", direction="out", labelsize=9, left=True, labelleft=True, bottom=False, labelbottom=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
# ax.axvline(x=6540.3, ymax=0.8, ls='--', color='k', alpha=0.5)  # noqa: ERA001
ax.axvline(x=6540.8, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.axvline(x=6543.0, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.axvline(x=6541.3, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.text(
    0.05,
    1.0,
    r"Mn(III) = 36.5 at.%",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    alpha=0.5,
)

# 图 C3
ax = subfig.add_subplot()
ax.set_position((left, 0.85, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.plot(Mn_2["Mn_2"]["Energy_Mn_2"], Mn_2["Mn_2"], ls="-", color=colors[4], label=r"0.2M Mn$\mathrm{^{2\!+}}$") # type: ignore
ax.plot(Mn_2["fit_Mn_II_1"]["Energy_fit"], Mn_2["fit_Mn_II_1"], ls="--", color=colors[1], label=None) # type: ignore

ax.set_xlim(6535.0, 6550.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylim(-0.01, 0.08)
# ax.set_ylabel(r'Relative Intensity (arb.u.)', fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.03, offset=-0.01))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.015, offset=-0.01))

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.7, 1.0),
    ncols=1,
    frameon=False,
    fontsize=9,
    labelcolor="linecolor",
    columnspacing=0.0,
)
ax.tick_params(
    axis="both", which="both", direction="out", labelsize=9, left=True, labelleft=True, bottom=False, labelbottom=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
# ax.axvline(x=6540.3, ymax=0.8, ls='--', color='k', alpha=0.5)  # noqa: ERA001
ax.axvline(x=6540.8, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.axvline(x=6543.0, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.axvline(x=6541.3, ymax=0.8, ls="--", color="k", alpha=0.5)

# 图 C4
ax = subfig.add_subplot()
ax.set_position((left, 1.23, 1.0, 1.0))
ax.set_box_aspect(0.3)

pad = 0.01
for i in range(opPreEdge.shape[1] // 4 - 6):
    ax.plot(
        opPreEdge.iloc[:, i * 4 + 24], opPreEdge.iloc[:, i * 4 + 27] + pad, c=xas_cmap[0].colors[i], ls="-", label=None # type: ignore
    )

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.04, 0.25, 0.05, 0.7)),
    location="right",
    orientation="vertical",
    cmap=xas_cmap[0].reversed(),
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False) # type: ignore
ax.text(
    1.1, 0.95, r"Discharge", transform=ax.transAxes, fontsize=9, va="top", ha="left", fontfamily="Arial", rotation=270
)

ax.set_xlim(6535.0, 6550.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylim(0, 0.08)
# ax.set_ylabel(r'Relative Intensity (arb.u.)', fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.03, offset=-0.01))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.015, offset=-0.01))

# ax.legend(loc='upper left', bbox_to_anchor=(0.7, 1.0), ncols=1, frameon=False, fontsize=9, labelcolor='linecolor', columnspacing=0.0)  # noqa: E501, ERA001
ax.tick_params(
    axis="both", which="both", direction="out", labelsize=9, left=True, labelleft=True, bottom=False, labelbottom=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
# ax.axvline(x=6540.3, ymax=0.8, ls='--', color='k', alpha=0.5)  # noqa: ERA001
ax.axvline(x=6540.8, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.axvline(x=6543.0, ymax=0.8, ls="--", color="k", alpha=0.5)
ax.axvline(x=6541.3, ymax=0.8, ls="--", color="k", alpha=0.5)

ax.text(
    -0.25,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_03_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_03_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_03_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_03_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure 2: The confirmations of 2e- transfers of α-MnO2 mechanism

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, data, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / data.axes_manager["x"].scale,
        "{} {}".format(size, data.axes_manager["x"].units),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=0.5,
        frameon=False,
        color=color,
        label_top=True,
    )
    ax.add_artist(asb)

In [ ]:
# opXANES 的数据
path_xas = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\Overview"
)
xas_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_xas, r"cell3_opXAS_P2a_Mn_Oct2022_Rkbg_1.3_Powder.nor"),
    sep=r"\s+",
    comment="#",
    header=None,
    index_col=None,
)
xas_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_xas, r"cell3_opXAS_P2a_Zn_Oct2022_Rkbg_1.1.nor"),
    sep=r"\s+",
    comment="#",
    header=None,
    index_col=None,
)

# 电池模型
cell_model = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\Model_cell.png"  # noqa: E501, RUF001
)

# Edge Jump 数据
path_ej = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\EdgeJump"  # noqa: RUF001
)
edgejump = pd.read_csv(Path.joinpath(path_ej, r"EdgeJump-V5_Dino.csv"), sep=",", comment="#", header=1, index_col=None)

# 读取 LCF, PCA, MCR 数据
path_all = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\FD1st\Oct2022_cell3_P2a\XANES"  # noqa: E501, RUF001
)
xas_mcr_Mn = xr.open_dataset(  # noqa: N816
    Path.joinpath(path_all, r"Mn", r"MCR", r"P2a_Mn_MCRHM_1reaction_comp_2_A_B.NETCDF4"), engine="h5netcdf"
)
xas_mcr_Zn = xr.open_dataset(  # noqa: N816
    Path.joinpath(path_all, r"Zn", r"MCR", r"P2a_Zn_MCRHM_1reaction_comp_2_A_B.NETCDF4"), engine="h5netcdf"
)
xas_lcf_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_all, r"Mn", r"LCF", r"cell3_1stFD_opXAS_P2a_Mn_LCF_powder.csv"),
    sep=",",
    comment="#",
    header=1,
    index_col=None,
)
xas_lcf_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_all, r"Zn", r"LCF", r"cell3_1stFD_opXAS_P2a_Zn_LCF.csv"),
    sep=",",
    comment="#",
    header=1,
    index_col=None,
)

In [ ]:
# 放电数据
charge_index = list(range(0, 14, 1))
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:3]  # noqa: N816
pdf_Mn.columns = [r"Energy", r"0.2M_MnSO4(aq.)", r"alpha_MnO2_Electrode"]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(3, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816
pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZSH"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# Edge Jump fitting
para_thin = np.polyfit(edgejump["MnO2"][:8], edgejump["ZHS"][:8], deg=1)
transform_thin = np.poly1d(para_thin)

In [ ]:
# 画图
fig = plt.figure(figsize=(7, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.0, figure=fig)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]] # type: ignore
labels = [[r"0.2M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
xas_colors = [
    ListedColormap(mpl.colormaps["sunset"](np.linspace(0.5, 1.0, opData_Mn.shape[1] - 1)), name="xas_colors_Mn"),
    ListedColormap(mpl.colormaps["sunset"](np.linspace(0.5, 0.0, opData_Zn.shape[1] - 1)), name="xas_colors_Zn"),
]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(pdf_Mn.shape[1] - 1):
    ax.plot(pdf_Mn.iloc[:, 0], pdf_Mn.iloc[:, i + 1], c=xascolors[0][i], ls="--", lw=1.0, label=labels[0][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Mn.shape[1] - 1):
    ax.plot(opData_Mn.iloc[:, 0], opData_Mn.iloc[:, i + 1], c=xas_colors[0].colors[i], ls="-", lw=1.0) # type: ignore

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.4, 0.07)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors[0],
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False) # type: ignore
ax.legend(loc="upper left", bbox_to_anchor=(0.51, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    0.50,
    0.2,
    "OCV",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.66,
    0.2,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    -0.21,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(pdf_Zn.shape[1] - 1):
    ax.plot(pdf_Zn.iloc[:, 0], pdf_Zn.iloc[:, i + 1], c=xascolors[1][i], ls="--", lw=1.0, label=labels[1][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Zn.shape[1] - 1):
    ax.plot(opData_Zn.iloc[:, 0], opData_Zn.iloc[:, i + 1], c=xas_colors[1].colors[i], ls="-", lw=1.0) # type: ignore

ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.tick_params(axis="x", labelsize=9)
ax.tick_params(axis="y", labelsize=9)
ax.legend(loc="upper left", bbox_to_anchor=(0.51, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.4, 0.07)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors[1],
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False) # type: ignore

ax.text(
    0.50,
    0.2,
    "OCV",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.66,
    0.2,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    -0.21,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.65, 0.0, 0.9, 0.9))
ax.set_box_aspect(1.0)

im = plt.imread(cell_model)
ax.imshow(im, aspect="equal")
ax.set_axis_off()
ax.text(
    -0.16,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

capacity = np.linspace(0, 270, 11)

ax.plot(
    capacity,
    xas_mcr_Mn["MCR_HM_C"][:, 1],
    label=r"MCR",
    c=xascolors[0][0],
    marker="o",
    markeredgecolor=xascolors[0][0],
    ls="-",
    markerfacecolor=xascolors[0][0],
)
# ax.plot(capacity, xas_mcr_Mn['MCR_HM_C_constrained'][:,1].data, label=None, c=xascolors[0][0], marker=None, ls='--') # noqa: E501, ERA001
ax.plot(
    capacity,
    xas_lcf_Mn["weight.1"].values, # type: ignore
    label=r"LCF",
    c=xascolors[0][0],
    marker="s",
    markeredgecolor=xascolors[0][0],
    ls="-",
    markerfacecolor="w",
)

ax.plot(
    capacity,
    xas_mcr_Mn["MCR_HM_C"][:, 0],
    label=None,
    c=xascolors[0][1],
    marker="o",
    markeredgecolor=xascolors[0][1],
    ls="-",
    markerfacecolor=xascolors[0][1],
)
# ax.plot(capacity, xas_mcr_Mn['MCR_HM_C_constrained'][:,0].data, label=None, c=xascolors[0][1], marker=None, ls='--')  # noqa: E501, ERA001
ax.plot(
    capacity,
    xas_lcf_Mn["weight"].values, # type: ignore
    label=None,
    c=xascolors[0][1],
    marker="s",
    markeredgecolor=xascolors[0][1],
    ls="-",
    markerfacecolor="w",
)

ax.set_xlim(-10.0, 300)
ax.set_xlabel(r"Capacity (mAh g$\mathrm{^{-1}_{MnO_2}}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=60))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=30))

ax.set_ylim(0.3, 0.8)
ax.set_ylabel(r"Mn Molar Fraction", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05))
ax.tick_params(axis="both", labelsize=9)

legend = ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.60, 0.99),
    ncols=1,
    frameon=False,
    labelcolor="k",
    fontsize=9,
    columnspacing=0.5,
    handletextpad=0.5,
    labelspacing=0.9,
)
legend.legend_handles[0].set_color("k") # type: ignore
legend.legend_handles[0].set_markerfacecolor("w") # type: ignore
legend.legend_handles[0].set_markeredgecolor("k") # type: ignore

legend.legend_handles[1].set_color("k") # type: ignore
legend.legend_handles[1].set_markerfacecolor("w") # type: ignore
legend.legend_handles[1].set_markeredgecolor("k")  # type: ignore

ax.text(
    0.30,
    0.94,
    f"{labels[0][0]}",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    c=xascolors[0][0],
)
ax.text(
    0.30,
    0.82,
    f"{labels[0][1]}",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    c=xascolors[0][1],
)
ax.text(
    -0.21,
    1.0,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(
    capacity,
    xas_mcr_Zn["MCR_HM_C"][:, 1],
    label=r"MCR",
    c=xascolors[1][1],
    marker="o",
    markeredgecolor=xascolors[1][1],
    ls="-",
    markerfacecolor=xascolors[1][1],
)
# ax.plot(capacity, xas_mcr_Zn['MCR_HM_C_constrained'][:,1].data, label=None, c=xascolors[1][1], marker=None, ls='--') # noqa: E501, ERA001
ax.plot(
    capacity,
    xas_lcf_Zn["weight"].values, # type: ignore
    label=r"LCF",
    c=xascolors[1][1],
    marker="s",
    markeredgecolor=xascolors[1][1],
    ls="-",
    markerfacecolor="w",
)

ax.plot(
    capacity,
    xas_mcr_Zn["MCR_HM_C"][:, 0],
    label=None,
    c=xascolors[1][0],
    marker="o",
    markeredgecolor=xascolors[1][0],
    ls="-",
    markerfacecolor=xascolors[1][0],
)
# ax.plot(capacity, xas_mcr_Zn['MCR_HM_C_constrained'][:,0].data, label=None, c=xascolors[1][0], marker=None, ls='--')  # noqa: E501, ERA001
ax.plot(
    capacity,
    xas_lcf_Zn["weight.1"].values, # type: ignore
    label=None,
    c=xascolors[1][0],
    marker="s",
    markeredgecolor=xascolors[1][0],
    ls="-",
    markerfacecolor="w",
)

ax.set_xlim(-10.0, 300)
ax.set_xlabel(r"Capacity (mAh g$\mathrm{^{-1}_{MnO_2}}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=60))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=30))

ax.set_ylim(-0.05, 1.2)
ax.set_ylabel(r"Zn Molar Fraction", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15))
ax.tick_params(axis="both", labelsize=9)

legend = ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.60, 0.99),
    ncols=1,
    frameon=False,
    labelcolor="k",
    fontsize=9,
    columnspacing=0.5,
    handletextpad=0.5,
    labelspacing=0.9,
)
legend.legend_handles[0].set_color("k") # type: ignore
legend.legend_handles[0].set_markerfacecolor("w") # type: ignore
legend.legend_handles[0].set_markeredgecolor("k") # type: ignore
legend.legend_handles[1].set_color("k") # type: ignore
legend.legend_handles[1].set_markerfacecolor("w") # type: ignore
legend.legend_handles[1].set_markeredgecolor("k") # type: ignore

ax.text(
    0.30,
    0.94,
    f"{labels[1][0]}",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    c=xascolors[1][0],
)
ax.text(
    0.30,
    0.82,
    f"{labels[1][1]}",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
    c=xascolors[1][1],
)
ax.text(
    -0.21,
    1.0,
    r"e",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)


# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(
    edgejump.iloc[:, 0],
    edgejump.iloc[:, 1],
    label=r"Edge Jump",
    c=colors[0], # type: ignore
    ls="-",
    marker="*",
    markeredgecolor=colors[0], # type: ignore
    markerfacecolor=colors[0], # type: ignore
    ms=10,
)
ax.plot(edgejump.iloc[:, 0], transform_thin(edgejump.iloc[:, 0]), label=None, c=colors[1], ls="--") # type: ignore

ax.set_xlabel(r"$\mathrm{Mn \ in \ {\alpha}}$-$\mathrm{MnO_2 \ (\mu mol)}$", fontsize=9)
ax.set_xlim(10.0, 3.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(r"$\mathrm{Zn \ in \ ZSH \ (\mu mol)}$", fontsize=9)
ax.set_ylim(-1, 20)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2.5))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)
# ax.legend(loc='upper left', bbox_to_anchor=(0.0, 1), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9) # noqa: E501, ERA001
# ax.text(0.05, 0.80, f'Slope: {abs(transform_thin[1]):.2f}', transform=ax.transAxes, fontsize=9, va='top', ha='left', fontfamily='Arial', )  # noqa: E501, ERA001
ax.text(
    -0.21,
    1.0,
    r"f",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_02_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_02_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_02_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_02_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)
plt.gcf().set_facecolor("white")
plt.show()

#### Figure 1: Capacity Limitations of α-MnO2 during discharge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bar, color, font_size=10):
    asb = AnchoredSizeBar(
        ax.transData,
        size / bar,
        "{} {}".format(size, "nm"),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=1,
        frameon=False,
        color=color,
        label_top=True,
        fontproperties={"size": font_size},
    )
    ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 电化学数据
path_echem = Path(r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\Echem\αMnO2")  # noqa: RUF001

gcd_kde = pd.read_excel(
    Path.joinpath(path_echem, r"GCD", r"1M ZnSO4 + 0.2M MnSO4", r"KDE-1M ZnSO4 + 0.2M MnSO4_all.xlsx"),
    sheet_name=0,
    index_col=None,
    header=0,
)
gcd_kde_no_mn = pd.read_excel(
    Path.joinpath(path_echem, r"GCD", r"1M ZnSO4", "KDE-1M ZnSO4_all.xlsx"), sheet_name=0, index_col=None, header=0
)

gcd_1st1st = pd.read_csv(
    Path.joinpath(
        path_echem, r"GCD", r"1M ZnSO4 + 1M MnSO4", r"GCD-01C-Zn-1M ZnSO4 + 1M MnSO4-αMnO2-1stDis_1stDis.csv"  # noqa: RUF001
    ),
    index_col=None,
    header=0,
)
gcd_002C1stDis = pd.read_csv(  # noqa: N816
    Path.joinpath(
        path_echem,
        r"GCD",
        r"1M ZnSO4 + 0.2M MnSO4",
        r"002C",
        r"Results",
        r"GCD-002C-Zn-1M ZnSO4 + 0.2M MnSO4-αMnO2.csv",  # noqa: RUF001
    ),
    index_col=None,
    header=0,
)

gcd_beakercell = pd.read_csv(
    Path.joinpath(
        path_echem,
        r"SpecialTest",
        r"PassivatedZnRichCoatting",
        r"pHBuffer",
        r"Bulk Cell",
        r"1M ZnSO4 + 0.2M MnSO4",
        r"LC-pH Buffer-1M ZnSO4 + 0.2M MnSO4-HAC-NaAC-4.19-αMnO2-nmg-12cm-8112",  # noqa: RUF001
        r"Results",
        r"LC-Zn-1M ZnSO4 + 0.2M MnSO4-HAC-NaAC-4.19-alphaMnO2-12.2mg-10_02_GCPL_C01_0.txt",
    ),
    sep=r"\s+",
    index_col=None,
    header=0,
)

# TEM 数据
tem_pristine = plt.imread(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2024-EMCA\TEM\Result\0005 - B8_Ceta_640_kx Ceta.png"  # noqa: E501, RUF001
)
tem_1stdischarge = plt.imread(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\TEM\Results\0018 - B8_Ceta_520_kx Ceta.png"  # noqa: E501, RUF001
)
path_eds_1stdischarge = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS" # noqa: E501, RUF001
)
eds_1stdischarge = hs.load(
    Path.joinpath(path_eds_1stdischarge, r"Data", r"0005 - B8_HAADF_135_kx", r"0005 - B8_HAADF_135_kx.emd")
) # type: ignore
eds_region = pd.read_csv(
    Path.joinpath(path_eds_1stdischarge, r"Results", r"0005 - B8_HAADF_135_kx", r"Sum", r"Hyperspy", r"EDX_Rrgion.csv"),
    sep=",",
    index_col=0,
    header=0,
)

In [ ]:
# 画图，双栏， 4.67 的宽度，三栏， 7.5 的高度  # noqa: RUF003
from matplotlib.lines import Line2D

fig = plt.figure(figsize=(4.67, 7.5))
gs = gridspec.GridSpec(3, 2, width_ratios=[1, 1], height_ratios=[1, 1, 1], wspace=0.0, hspace=0.0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0.0, 0.9, 0.9))
ax.set_box_aspect(0.8)

x, y = gcd_kde["loadings (mg/cm2)"], gcd_kde["capacity(mA h/g)"]
sns.scatterplot(
    data=gcd_kde,
    x=x,
    y=y,
    s=100,
    linewidth=1,
    edgecolor="gainsboro",
    ax=ax,
    hue="ratios",
    palette=[colors[1], colors[3]], # type: ignore
    legend=True,
)
# sns.kdeplot(data=gcd_kde, x=x, y=y, ax=ax, zorder=5,  levels=6, hue="ratios", fill=False, legend=False, palette=[colors[1], colors[3]], alpha=0.3)  # noqa: E501, ERA001

x, y = gcd_kde_no_mn["loadings (mg/cm2)"], gcd_kde_no_mn["capacity(mA h/g)"]
sns.scatterplot(
    data=gcd_kde_no_mn,
    x=x,
    y=y,
    s=100,
    linewidth=1,
    edgecolor="k",
    ax=ax,
    hue="ratios",
    palette=[colors[1], colors[3]], # type: ignore
    legend=True,
    alpha=0.8,
)
# sns.kdeplot(data=gcd_kde_no_mn, x=x, y=y, ax=ax, zorder=5,  levels=3, hue="ratios", fill=False, legend=False, palette=['k', 'k'], alpha=0.6)  # noqa: E501, ERA001

ax.set_xlabel(r"Mass Loadings (mg cm$^{-2}$)", fontsize=9)
ax.set_xlim(0.5, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: "%.1f" % x))  # noqa: ARG005
ax.tick_params(axis="both", labelsize=9)
ax.set_ylabel(r"Capacity (mAh g$\mathrm{{^{-1}_{MnO_2}}}$)", fontsize=9)
ax.set_ylim(50, 650)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=100, offset=-50))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=50, offset=-50))
ax.tick_params(axis="y", labelsize=9)
ax.axhline(y=308.0, xmin=0.0, xmax=1.0, ls="--", lw=1.0, c="silver")
ax.axhline(y=616.0, xmin=0.0, xmax=1.0, ls="--", lw=1.0, c="silver")
ax.text(
    0.98, 0.38, r"1 e-", transform=ax.transAxes, fontsize=9, va="center", ha="right", fontfamily="Arial", color="silver"
)
ax.text(
    0.98, 0.89, r"2 e-", transform=ax.transAxes, fontsize=9, va="center", ha="right", fontfamily="Arial", color="silver"
)

# 修改 legend 的 label
legend_labels = [
    r"$\mathrm{70\alpha}$-$\mathrm{MnO_2}$-$\mathrm{30CNTs}$",
    r"$\mathrm{80\alpha}$-$\mathrm{MnO_2}$-$\mathrm{9SuperP}$-$\mathrm{9PVdf}$-$\mathrm{2CNTs}$",
]
legend1_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label=legend_labels[0],
        markerfacecolor=colors[1], # type: ignore
        markeredgecolor="none",
        markersize=8,
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label=legend_labels[1],
        markerfacecolor=colors[3], # type: ignore
        markeredgecolor="none",
        markersize=8,
    ),
]
legend1 = ax.legend(
    handles=legend1_handles,
    loc="upper right",
    bbox_to_anchor=(0.85, 0.97),
    ncols=1,
    frameon=False,
    fontsize=7,
    handlelength=1.0,
    handletextpad=0.4,
)
ax.add_artist(legend1)  # 保留第一个图例

legend2_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label=r"$\mathrm{Mn(II) \ additive}$",
        markerfacecolor="w",
        markeredgecolor="gainsboro",
        markersize=8,
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label=r"$\mathrm{No \ Mn(II) \ additive}$",
        markerfacecolor="w",
        markeredgecolor="k",
        markersize=8,
    ),
]
legend2 = ax.legend(
    handles=legend2_handles,
    loc="lower right",
    bbox_to_anchor=(1.0, 0.54),
    fontsize=7,
    frameon=False,
    handlelength=1.0,
    handletextpad=0.4,
)

ax.text(
    -0.25,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0, 0.9, 0.9))
ax.set_box_aspect(0.8)

ax.plot(gcd_002C1stDis.iloc[:, 0], gcd_002C1stDis.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label=r"C/50") # type: ignore
ax.plot(gcd_1st1st.iloc[:, 1], gcd_1st1st.iloc[:, 0], ls="-", lw=1.0, c=colors[1], label=r"C/10") # type: ignore
ax.plot(gcd_1st1st.iloc[:, 3], gcd_1st1st.iloc[:, 2], ls="-", lw=1.0, c=colors[1], label=None) # type: ignore
ax.plot(
    gcd_beakercell.iloc[:, 2] / 0.00065,
    gcd_beakercell.iloc[:, 1],
    ls="-",
    lw=1.0,
    c=colors[2], # type: ignore
    label=r"Bulk Cell, C/10",
)

ax.set_xlabel(r"Capacity (mAh g$\mathrm{^{-1}_{MnO_2}}$)", fontsize=9)
ax.set_xlim(0, 350)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=100, offset=50))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=50, offset=50))
ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_ylim(0.8, 1.6)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1))
ax.set_xticks(ticks=[0, 50, 150, 250, 350], labels=["0", "50", "150", "250", "350"])
ax.tick_params(axis="both", labelsize=9)

ax.legend(
    loc="upper right",
    bbox_to_anchor=(1, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handlelength=1.0,
    handletextpad=0.4,
)
ax.text(
    0.15,
    0.2,
    r"$\mathrm{2^{nd} \ Discharge}$",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="left",
    fontfamily="Arial",
    c=colors[1], # type: ignore
)
ax.text(
    0.15,
    0.10,
    r"$\mathrm{with \ fresh \ electrolyte}$",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="left",
    fontfamily="Arial",
    c=colors[1], # type: ignore
)

ax.text(
    -0.25,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.2, 0.1, 0.9, 0.9))
ax.set_box_aspect(1.0)

img = ndimage.rotate(tem_pristine, -10, reshape=True)
ax.imshow(img[2850:3650, 1300:2500], cmap="gray", vmin=0.0, vmax=0.25, alpha=1.0)
# ax.imshow(img[2850:4050, 1300:2500], cmap='gray', vmin=0.0, vmax=0.25, alpha=1.0)  # noqa: ERA001
sizebar = add_sizebar(ax, 1, 0.00954, "w", font_size=11)
sizebar.set_bbox_to_anchor((0.01, 0.15), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.text(
    -0.03,
    0.85,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.2, 0.48, 0.9, 0.9))
ax.set_box_aspect(1.0)

img = ndimage.rotate(tem_1stdischarge, 90, reshape=True)
ax.imshow(img[800:1150, 300:800], cmap="gray", aspect=1.0, vmin=0.0, vmax=0.70, alpha=1.0)
# ax.imshow(img[800:1300, 300:800], cmap='gray', aspect=1.0, vmin=0.0, vmax=0.70, alpha=1.0)  # noqa: ERA001
sizebar = add_sizebar(ax, 1, 0.0237, "w", font_size=11)
sizebar.set_bbox_to_anchor((0.01, 0.15), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.text(
    -0.03,
    0.85,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E1
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.17, 0.24, 0.8, 0.8))
ax.set_box_aspect(0.5)

vmax = 1000 * (eds_1stdischarge[-2].max(-1).max(-1).data[0] // 1000)  # 需要做调整
im = ax.imshow(eds_1stdischarge[-2].data, cmap="gray", vmin=0.0, vmax=vmax)
scalebar = add_sizebar(ax, 50, eds_1stdischarge[-2].axes_manager[0].scale, "w", font_size=9)
scalebar.set_bbox_to_anchor((0.1, 0), transform=ax.transAxes) # type: ignore
ax.set_axis_off()

ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)
ax.text(
    0.85, 0.02, r"HAADF", transform=ax.transAxes, fontsize=9, va="bottom", ha="right", fontfamily="Arial", color="w"
)

ax.text(
    0.08, 1.0, r"e", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 E2
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.48, 0.24, 0.8, 0.8))
ax.set_box_aspect(0.5)

vmax = eds_1stdischarge[4].max(-1).max(-1).data[0] // 1  # 需要做调整
im = ax.imshow(
    eds_1stdischarge[4].data,
    cmap=transparent_single_color_cmap(color0="k", color1=(26 / 255, 51 / 255, 203 / 255)),
    vmin=0.0,
    vmax=vmax,
)
scalebar = add_sizebar(ax, 50, eds_1stdischarge[4].axes_manager[0].scale, "w", font_size=9)
scalebar.set_bbox_to_anchor((0.1, 0), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)
ax.text(
    0.85,
    0.02,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)

# 图 E3
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.17, -0.17, 0.8, 0.8))
ax.set_box_aspect(0.5)

vmax = 10 * (eds_1stdischarge[5].max(-1).max(-1).data[0] // 10 + 1)  # 需要做调整
im = ax.imshow(
    eds_1stdischarge[5].data,
    cmap=transparent_single_color_cmap(color0="k", color1=(236 / 255, 236 / 255, 0)),
    vmin=0.0,
    vmax=vmax,
)
scalebar = add_sizebar(ax, 50, eds_1stdischarge[5].axes_manager[0].scale, "w", font_size=9)
scalebar.set_bbox_to_anchor((0.1, 0), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)
ax.text(
    0.85,
    0.02,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)

# 图 E4
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.48, -0.17, 0.8, 0.8))
ax.set_box_aspect(0.5)

vmax = eds_1stdischarge[6].max(-1).max(-1).data[0] // 1 + 1  # 需要做调整
im = ax.imshow(
    eds_1stdischarge[6].data,
    cmap=transparent_single_color_cmap(color0="k", color1=(252 / 255, 0, 252 / 255)),
    vmin=0.0,
    vmax=vmax,
)
scalebar = add_sizebar(ax, 50, eds_1stdischarge[6].axes_manager[0].scale, "w", font_size=9)
scalebar.set_bbox_to_anchor((0.1, 0), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)
ax.text(
    0.85,
    0.02,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)

# 图 E5
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.17, -0.58, 0.8, 0.8))
ax.set_box_aspect(0.5)

vmax = eds_1stdischarge[9].max(-1).max(-1).data[0] // 1 + 1  # 需要做调整
im = ax.imshow(
    eds_1stdischarge[9].data,
    cmap=transparent_single_color_cmap(color0="k", color1=(0 / 255, 255 / 255, 0 / 255)),
    vmin=0.0,
    vmax=vmax,
)
scalebar = add_sizebar(ax, 50, eds_1stdischarge[9].axes_manager[0].scale, "w", font_size=9)
scalebar.set_bbox_to_anchor((0.1, 0), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)
ax.text(
    0.85,
    0.02,
    r"O $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)

# 图 E6
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.48, -0.58, 0.8, 0.8))
ax.set_box_aspect(0.5)

vmax = eds_1stdischarge[7].max(-1).max(-1).data[0] // 1  # 需要做调整
im = ax.imshow(
    eds_1stdischarge[7].data,
    cmap=transparent_single_color_cmap(color0="k", color1=(0 / 255, 245 / 255, 245 / 255)),
    vmin=0.0,
    vmax=vmax,
)
scalebar = add_sizebar(ax, 50, eds_1stdischarge[7].axes_manager[0].scale, "w", font_size=9)
scalebar.set_bbox_to_anchor((0.1, 0), transform=ax.transAxes) # type: ignore
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)
ax.text(
    0.85,
    0.02,
    r"S $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    color="w",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_01_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_01_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_01_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_Figure_01_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)
plt.gcf().set_facecolor("white")
plt.show()